# Installing Libraries

In [ ]:
!pip uninstall -y transformers torchvision torch torchaudio accelerate -q

!pip install -q \
torch==2.7.1 \
torchvision==0.22.1 \
torchaudio==2.7.1 \
--index-url https://download.pytorch.org/whl/cu128

!pip install -q \
transformers==4.53.3 \
accelerate==1.8.1 \
datasets==4.0.0 \
evaluate==0.4.5 \
sentencepiece

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
peft 0.19.1 requires accelerate>=0.21.0, which is not installed.
peft 0.19.1 requires transformers, which is not installed.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Imports
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

warnings.filterwarnings("ignore")

In [ ]:
#GPU SetUp
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device : cuda
Tesla T4


In [ ]:
#Seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
#Cuda Optimization
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Cross Validation/Best_XLMR_MLM8"
SAVE_DIR = "/content/drive/MyDrive/Cross Validation/XLMR_CV1"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
MAX_LEN = 64

In [ ]:
# Tokenizer Function
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

In [ ]:
# Verify
print(tokenizer("Climate change is real."))

{'input_ids': [0, 135987, 67, 15549, 83, 2773, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
# Load Dataset
english_df = pd.read_csv("/content/English_train_data.csv")
hindi_df = pd.read_csv("/content/english_to_hindi.csv")
bengali_df = pd.read_csv("/content/english_to_bengali.csv")
none_df = pd.read_csv("/content/non_cliamte_none_data.csv")
none_hindi_df = pd.read_csv("/content/english_to_hindi_nonclimate.csv")
none_bengali_df = pd.read_csv("/content/english_to_bengali_nonclimate.csv")

In [ ]:
none_df = none_df.head(3300)
none_bengali_df = none_bengali_df.head(3300)
none_hindi_df = none_hindi_df.head(3300)

In [ ]:
hindi = hindi_df[["hindi","label"]].copy()
hindi.columns = ["sentence","label"]

bengali = bengali_df[["bengali","label"]].copy()
bengali.columns = ["sentence","label"]

none_hindi = none_hindi_df[["hindi","label"]].copy()
none_hindi.columns = ["sentence","label"]

none_bengali = none_bengali_df[["bengali","label"]].copy()
none_bengali.columns = ["sentence","label"]

english = english_df[["sentence","label"]].copy()

none_df = none_df[["sentence","label"]].copy()


In [ ]:
train_df = pd.concat(
    [
        english,
        hindi,
        bengali,
        none_df,
        none_hindi,
        none_bengali
    ],

    ignore_index=True

)

In [ ]:
train_df.shape

(48213, 2)

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48213 entries, 0 to 48212
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sentence  48211 non-null  object
 1   label     48213 non-null  object
dtypes: object(2)
memory usage: 753.5+ KB


In [ ]:
# Remove Missing Values
train_df = train_df.dropna()
train_df = train_df.reset_index(drop=True)

In [ ]:
# Remove Duplicate Sentences
train_df = train_df.drop_duplicates(subset="sentence")
train_df = train_df.reset_index(drop=True)

In [ ]:
# Shuffle Dataset
train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [ ]:
# Check Dataset
print(train_df.head())

                                            sentence    label
0  : मिलॉय: अब तक के सबसे मूर्खतापूर्ण ग्लोबल वार...  AGAINST
1  সম্পর্কহীন নয় কিন্তু বিশেষভাবে পুনরায়: জলবায...     NONE
2  কার্বন নির্গমন 2014 সালে সর্বোচ্চ ছিল না এবং 2...     NONE
3                                जलवायु परिवर्तन संघ  AGAINST
4               Climate change is a global emergency    FAVOR


In [ ]:
# Label Encoding
label2id = {
    "AGAINST":0,
    "FAVOR":1,
    "NONE":2
}
id2label = {
    0:"AGAINST",
    1:"FAVOR",
    2:"NONE"
}
train_df["label"] = train_df["label"].map(label2id)

In [ ]:
# Verify labels
print(train_df["label"].value_counts())
print()
print(train_df.head())

label
2    22421
0    12610
1    12605
Name: count, dtype: int64

                                            sentence  label
0  : मिलॉय: अब तक के सबसे मूर्खतापूर्ण ग्लोबल वार...      0
1  সম্পর্কহীন নয় কিন্তু বিশেষভাবে পুনরায়: জলবায...      2
2  কার্বন নির্গমন 2014 সালে সর্বোচ্চ ছিল না এবং 2...      2
3                                जलवायु परिवर्तन संघ      0
4               Climate change is a global emergency      1


In [ ]:
# Check Class Balance
print(train_df["label"].value_counts(normalize=True))

label
2    0.470673
0    0.264716
1    0.264611
Name: proportion, dtype: float64


In [ ]:
# Sentence Length Analysis
lengths = train_df["sentence"].apply(
    lambda x: len(tokenizer.tokenize(str(x)))
)
print(lengths.describe())

Token indices sequence length is longer than the specified maximum sequence length for this model (1491 > 512). Running this sequence through the model will result in indexing errors


count    47636.000000
mean        32.810144
std         32.532883
min          1.000000
25%         21.000000
50%         29.000000
75%         38.000000
max       1603.000000
Name: sentence, dtype: float64


In [ ]:
# Check Maximum Length
print("Maximum Tokens :", lengths.max())
print("95 Percentile :", np.percentile(lengths,95))

Maximum Tokens : 1603
95 Percentile : 61.0


In [ ]:
# Final Dataset Information
print("="*60)
print("Total Samples :", len(train_df))
print()
print(train_df["label"].value_counts())
print("="*60)

Total Samples : 47636

label
2    22421
0    12610
1    12605
Name: count, dtype: int64


In [ ]:
# Clean and Save Dataset
train_df.to_csv(
    "/content/drive/MyDrive/clean_train.csv",
    index=False
)

PART 3 — 5-Fold Stratified Cross Validation

In [ ]:
# Stratified 5-Fold Cross Validation
NUM_FOLDS = 5

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

In [ ]:
# Create Metric Lists
fold_accuracy = []
fold_precision = []
fold_recall = []
fold_f1 = []

In [ ]:
# Reset MODEL_PATH to the base pre-trained model for the CV loop
MODEL_PATH = "/content/drive/MyDrive/Cross Validation/Best_XLMR_MLM8"
print(f"MODEL_PATH reset to: {MODEL_PATH}")

MODEL_PATH reset to: /content/drive/MyDrive/Cross Validation/Best_XLMR_MLM8


In [ ]:
# Start Cross Validation Loop
for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df["sentence"], train_df["label"])):

    print("="*70)
    print(f"Fold {fold+1}/{NUM_FOLDS}")
    print("="*70)

    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    print("Training Samples :", len(train_fold))
    print("Validation Samples :", len(val_fold))

Fold 1/5
Training Samples : 38108
Validation Samples : 9528
Fold 2/5
Training Samples : 38109
Validation Samples : 9527
Fold 3/5
Training Samples : 38109
Validation Samples : 9527
Fold 4/5
Training Samples : 38109
Validation Samples : 9527
Fold 5/5
Training Samples : 38109
Validation Samples : 9527


In [ ]:
# Compute Class Weights inside every fold
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_fold["label"]),
    y=train_fold["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print(class_weights)

tensor([1.2592, 1.2597, 0.7082], device='cuda:0')


In [ ]:
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(
    train_fold,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_fold,
    preserve_index=False
)

In [ ]:
# Tokenize
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/38109 [00:00<?, ? examples/s]

Map:   0%|          | 0/9527 [00:00<?, ? examples/s]

In [ ]:
# Remove Sentence Column
train_dataset = train_dataset.remove_columns(
    ["sentence"]
)

val_dataset = val_dataset.remove_columns(
    ["sentence"]
)

In [ ]:
# Rename Label Column
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label",
    "labels"
)

In [ ]:
# Torch Format
train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [ ]:
# Check Dataset
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 38109
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 9527
})


PART 4 — Fine-Tuning with Cross Validation

In [ ]:
# Data Collator
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

In [ ]:
# Metrics Function
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Weighted Trainer
from transformers import Trainer

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights
        )

        loss = loss_fn(
            logits,
            labels
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./Results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    save_total_limit=1,
    report_to="none",
    seed=42
)

In [ ]:
# Early Stopping
early_stop = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001
)

In [ ]:
# Load a Fresh Model for Each Fold
config = AutoConfig.from_pretrained(MODEL_PATH)
config.num_labels = 3
config.label2id = label2id
config.id2label = id2label
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=config,
    ignore_mismatched_sizes=True
)
model.to(device)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/Cross Validation/Best_XLMR_MLM8 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [ ]:
# Freeze Embeddings
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False

In [ ]:
# Start Full Cross Validation Loop (including training, evaluation, and saving)
for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df["sentence"], train_df["label"])):

    print("="*70)
    print(f"Fold {fold+1}/{NUM_FOLDS}")
    print("="*70)

    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    print("Training Samples :", len(train_fold))
    print("Validation Samples :", len(val_fold))

    # Compute Class Weights inside every fold
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_fold["label"]),
        y=train_fold["label"]
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float
    ).to(device)

    print("Class Weights for Fold ", fold+1, ":", class_weights)

    # Convert to HuggingFace Dataset
    train_dataset = Dataset.from_pandas(
        train_fold,
        preserve_index=False
    )

    val_dataset = Dataset.from_pandas(
        val_fold,
        preserve_index=False
    )

    # Tokenize
    train_dataset = train_dataset.map(
        tokenize,
        batched=True
    )

    val_dataset = val_dataset.map(
        tokenize,
        batched=True
    )

    # Remove Sentence Column
    train_dataset = train_dataset.remove_columns(
        ["sentence"]
    )

    val_dataset = val_dataset.remove_columns(
        ["sentence"]
    )

    # Rename Label Column
    train_dataset = train_dataset.rename_column(
        "label",
        "labels"
    )

    val_dataset = val_dataset.rename_column(
        "label",
        "labels"
    )

    # Torch Format
    train_dataset.set_format("torch")
    val_dataset.set_format("torch")

    # Check Dataset (optional, for verification within the loop)
    print("Train Dataset for Fold ", fold+1, ":", train_dataset)
    print("Validation Dataset for Fold ", fold+1, ":", val_dataset)

    # Load a Fresh Model for Each Fold
    config = AutoConfig.from_pretrained(MODEL_PATH)
    config.num_labels = 3
    config.label2id = label2id
    config.id2label = id2label

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        config=config,
        ignore_mismatched_sizes=True
    )
    model.to(device)

    # Freeze Embeddings
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

    # Create Trainer
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[early_stop]
    )

    # Train
    print(f"\nStarting training for Fold {fold+1}...")
    trainer.train()
    print(f"Finished training for Fold {fold+1}.")

    # Evaluate
    print(f"\nStarting evaluation for Fold {fold+1}...")
    metrics = trainer.evaluate()
    print(f"Evaluation Metrics for Fold {fold+1}: {metrics}")

    fold_accuracy.append(metrics["eval_accuracy"])
    fold_precision.append(metrics["eval_precision"])
    fold_recall.append(metrics["eval_recall"])
    fold_f1.append(metrics["eval_f1"])

    # Save Each Fold
    fold_dir = os.path.join(
        SAVE_DIR,
        f"Fold_{fold+1}"
    )

    trainer.save_model(fold_dir)
    tokenizer.save_pretrained(fold_dir)

    print(f"Model Saved -> {fold_dir}\n")

print("="*70)
print("Cross-validation loop completed.")
print("="*70)

Fold 1/5
Training Samples : 38108
Validation Samples : 9528
Class Weights for Fold  1 : tensor([1.2592, 1.2597, 0.7082], device='cuda:0')


Map:   0%|          | 0/38108 [00:00<?, ? examples/s]

Map:   0%|          | 0/9528 [00:00<?, ? examples/s]

Train Dataset for Fold  1 : Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 38108
})
Validation Dataset for Fold  1 : Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 9528
})


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/Cross Validation/Best_XLMR_MLM8 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/Cross Validation/Best_XLMR_MLM8 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting training for Fold 5...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.801800,0.596851,0.750184,0.739430,0.757521,0.743394
2,0.542700,0.552023,0.764774,0.758262,0.778840,0.759948
3,0.395000,0.531621,0.795004,0.788052,0.802228,0.791111
4,0.286000,0.560677,0.808124,0.797855,0.817938,0.804658
5,0.209100,0.613431,0.815787,0.806053,0.822324,0.812066


Finished training for Fold 5.

Starting evaluation for Fold 5...


Evaluation Metrics for Fold 5: {'eval_loss': 0.6134308576583862, 'eval_accuracy': 0.8157867114516637, 'eval_precision': 0.8060525625488214, 'eval_recall': 0.8223242130069345, 'eval_f1': 0.812066105882769, 'eval_runtime': 9.9599, 'eval_samples_per_second': 956.536, 'eval_steps_per_second': 29.92, 'epoch': 5.0}
Model Saved -> /content/drive/MyDrive/Cross Validation/XLMR_CV1/Fold_5

Cross-validation loop completed.


In [ ]:
# Cross Validation Summary
results_df = pd.DataFrame({
    "Fold": range(1, NUM_FOLDS + 1),
    "Accuracy": fold_accuracy,
    "Precision": fold_precision,
    "Recall": fold_recall,
    "Macro_F1": fold_f1
})

print(results_df)

   Fold  Accuracy  Precision    Recall  Macro_F1
0     1  0.827771   0.817971  0.830986  0.823395
1     2  0.812743   0.803238  0.817565  0.808527
2     3  0.809174   0.801069  0.820997  0.807152
3     4  0.805395   0.797042  0.816760  0.802671
4     5  0.815787   0.806053  0.822324  0.812066


In [ ]:
# Average Performance
print("="*70)
print("5-Fold Cross Validation Results")
print("="*70)

print(f"Average Accuracy : {np.mean(fold_accuracy):.4f}")
print(f"Average Precision: {np.mean(fold_precision):.4f}")
print(f"Average Recall   : {np.mean(fold_recall):.4f}")
print(f"Average Macro F1 : {np.mean(fold_f1):.4f}")

print()

print(f"Std Accuracy : {np.std(fold_accuracy):.4f}")
print(f"Std Macro F1 : {np.std(fold_f1):.4f}")

5-Fold Cross Validation Results
Average Accuracy : 0.8142
Average Precision: 0.8051
Average Recall   : 0.8217
Average Macro F1 : 0.8108

Std Accuracy : 0.0076
Std Macro F1 : 0.0070


In [ ]:
# Find the Best Fold
best_fold = np.argmax(fold_f1)

print(f"\nBest Fold : {best_fold+1}")
print(f"Best Macro F1 : {fold_f1[best_fold]:.4f}")


Best Fold : 1
Best Macro F1 : 0.8234


In [ ]:
# Copy the Best Model
import shutil

best_model = os.path.join(
    SAVE_DIR,
    f"Fold_{best_fold+1}"
)

final_model = os.path.join(
    SAVE_DIR,
    "Best_Model"
)

if os.path.exists(final_model):
    shutil.rmtree(final_model)

shutil.copytree(best_model, final_model)

print("Best model copied successfully!")

Best model copied successfully!


In [ ]:
# Load Best Model
BEST_MODEL = os.path.join(SAVE_DIR, "Best_Model")
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL)
model.to(device)
model.eval()
print("Best model loaded successfully.")

Best model loaded successfully.


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# Load Best CV Model
# ==========================================

MODEL_PATH = "/content/drive/MyDrive/Cross Validation/XLMR_CV1/Best_Model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

print("Model Loaded Successfully!")

# ==========================================
# Label Mapping
# ==========================================

label_map = {
    0: "AGAINST",
    1: "FAVOR",
    2: "NONE"
}

# ==========================================
# Prediction Function
# ==========================================

def predict(text):

    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():

        outputs = model(**inputs)

        pred = torch.argmax(outputs.logits, dim=1).item()

    return label_map[pred]

Model Loaded Successfully!


In [ ]:
hindi_test = pd.read_excel("/content/Hindi_500_test data.xlsx")

print(hindi_test.head())

hindi_test["Predicted_Label"] = hindi_test["COMMENT"].apply(predict)

hindi_test.to_excel(
    "/content/Hindi_500_Predictions_CV2X.xlsx",
    index=False
)

print("Hindi Prediction Saved Successfully!")

   ID                                            COMMENT
0   1  बिल नाई एक वैज्ञानिक नहीं हैं। वह एक यांत्रिक ...
1   2           ऑनलाइन कक्षाओं के लिए और कौन उपस्थित है?
2   3  शाकाहारियों को इतना परेशान करने वाला क्यों होत...
3   4                      विवरण में बिल न्ये मर्च लिंक।
4   5  बिल के प्रति इतना घृणा क्यों है? मुझे समझ नहीं...
Hindi Prediction Saved Successfully!


In [ ]:
bangla_test = pd.read_excel("/content/Bangla_500_test data.xlsx")

print(bangla_test.head())

bangla_test["Predicted_Label"] = bangla_test["COMMENT"].apply(predict)

bangla_test.to_excel(
    "/content/Bangla_500_Predictions_CV2X.xlsx",
    index=False
)

print("Bangla Prediction Saved Successfully!")

   ID                                            COMMENT
0   1  পৃথিবীর নৃত্য এবং অনন্ত আকাশের মধ্যে, ফিসফিস ক...
1   2  ধন্যবাদ, আপনার করা সমস্ত কিছুর জন্য এবং আপনি য...
2   3  এটি বায়ুমণ্ডলে কার্বন ডাই-অক্সাইডের ২৫০ ডিগ্র...
3   4  জলবায়ু-পরিবর্তক দেবতা-পাথর জানে যে পৃথিবীর দু...
4   5  আমার শিক্ষক আমাকে বলেছিলেন যে প্রতি ১০০ বছরে প...
Bangla Prediction Saved Successfully!
